In [ ]:
#https://www.slingacademy.com/article/implementing-multivariate-forecasting-using-grus-in-pytorch/

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import csv
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

train_raw = pd.read_csv('train.csv')
test_raw = pd.read_csv('test.csv')

def Insert_Dates(df):
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['hour'] = df['date'].dt.hour

    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)


    return df

train_raw = Insert_Dates(train_raw)
test_raw = Insert_Dates(test_raw)

test_raw['OT'] = 0.0

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_raw[['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']])
test_scaled = scaler.transform(test_raw[['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']])


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

def create_sequences(data, seq_length):
    xs = []
    ys = []

    # Last column is oil temperature
    for i in range(len(data) - seq_length):
        x = data[i:i + seq_length, :-1] # All features except OT
        y = data[i + seq_length, -1]
        xs.append(x)
        ys.append(y)

    return np.array(xs), np.array(ys)

class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim

        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True, dropout=0.2, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        # Initialize hidden state
        h0 = torch.zeros(self.layer_dim * 2, x.size(0), self.hidden_dim, device=x.device).requires_grad_()

        # Forward pass
        out, hn = self.gru(x, h0.detach())

        # Reshape
        out = self.fc(out[:, -1, :])
        return out

training_sequences = create_sequences(train_scaled, 1)
test_sequences = create_sequences(test_scaled, 1)

# Generate sequences
SEQ_LEN = 96
train_x, train_y = create_sequences(train_scaled, SEQ_LEN)

# Build combined data
combined_scaled = np.vstack([train_scaled, test_scaled])

# Build test sequences using the last 24 rows of train
test_x = []
for i in range(len(train_scaled) - SEQ_LEN, len(combined_scaled) - SEQ_LEN):
    seq = combined_scaled[i : i + SEQ_LEN, :-1]
    test_x.append(seq)

test_x = np.array(test_x)


# Convert to GRU-friendly shape (batch, seq_len=1, features)
tensor_train_features = torch.Tensor(train_x).to(device)
tensor_train_labels   = torch.Tensor(train_y).unsqueeze(1).to(device)
tensor_test           = torch.Tensor(test_x).to(device)

# Put it into dataset
dataset = TensorDataset(tensor_train_features, tensor_train_labels)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Define model
model = GRUModel(input_dim=6, hidden_dim=256, layer_dim=3, output_dim=1).to(device)
criterion = nn.HuberLoss(delta=1.0) 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_model(model, dataloader, criterion, optimizer, epochs=30):
    for epoch in  range(epochs):
        for i, (features, labels) in enumerate(dataloader):
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if(i + 1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(dataloader)}], Loss: {loss.item():.4f}')

train_model(model, dataloader, criterion, optimizer)

model.eval()
with torch.no_grad():
    preds_scaled = model(tensor_test).cpu().numpy()

preds_full = np.zeros((preds_scaled.shape[0], train_scaled.shape[1]))
preds_full[:, -1] = preds_scaled[:, 0]

# Now inverse transform
preds_real = scaler.inverse_transform(preds_full)[:, -1]  # take only OT


# Save CSV
with open("test_predictions.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "OT"])
    for i, val in enumerate(preds_real):
        writer.writerow([i + 14000, float(val)])

Device:  cuda
Epoch [1/30], Step [10/218], Loss: 0.0190
Epoch [1/30], Step [20/218], Loss: 0.0145
Epoch [1/30], Step [30/218], Loss: 0.0133
Epoch [1/30], Step [40/218], Loss: 0.0111
Epoch [1/30], Step [50/218], Loss: 0.0118
Epoch [1/30], Step [60/218], Loss: 0.0144
Epoch [1/30], Step [70/218], Loss: 0.0087
Epoch [1/30], Step [80/218], Loss: 0.0164
Epoch [1/30], Step [90/218], Loss: 0.0090
Epoch [1/30], Step [100/218], Loss: 0.0098
Epoch [1/30], Step [110/218], Loss: 0.0156
Epoch [1/30], Step [120/218], Loss: 0.0098
Epoch [1/30], Step [130/218], Loss: 0.0117
Epoch [1/30], Step [140/218], Loss: 0.0141
Epoch [1/30], Step [150/218], Loss: 0.0189
Epoch [1/30], Step [160/218], Loss: 0.0099
Epoch [1/30], Step [170/218], Loss: 0.0102
Epoch [1/30], Step [180/218], Loss: 0.0126
Epoch [1/30], Step [190/218], Loss: 0.0119
Epoch [1/30], Step [200/218], Loss: 0.0112
Epoch [1/30], Step [210/218], Loss: 0.0129
Epoch [2/30], Step [10/218], Loss: 0.0141
Epoch [2/30], Step [20/218], Loss: 0.0115
Epoch [2